# Predicting the Sale Price of Bulldozers using Machine Learning
> The goal of the project is to predict the sale price of a particular piece of heavy equiment at auction based on it's usage, equipment type, and configuaration.  The data is sourced from auction result postings and includes information on usage and equipment configurations.


## Modelling

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_log_error, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

### Building a custom Transformers for the pipeline for Preprocessing and preparing the data for modelling

In [24]:
# Create a pipeline that extracts year, month, day, DayOfWeek from the saledate column and then drop the saledate column
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Extracts date components from saledate and drops the original.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if "saledate" in X.columns:
            X["saledate"] = pd.to_datetime(X["saledate"])
            X["saleYear"] = X["saledate"].dt.year
            X["saleMonth"] = X["saledate"].dt.month
            X["saleDayOfWeek"] = X["saledate"].dt.dayofweek
            X["saleDayOfYear"] = X["saledate"].dt.dayofyear
            X.drop("saledate", axis=1, inplace=True)

        return X

class TabularImputerAndEncoder(BaseEstimator, TransformerMixin):
    """
    Learns training medians and category mappings, then applies them to any raw dataset.
    """
    def __init__(self):
        self.training_medians_ = {}
        self.category_mappings_ = {}

    def fit(self, X, y=None):
        X = X.copy()
        # Learn numeric medians
        for label, content in X.items():
            if pd.api.types.is_numeric_dtype(content):
                if pd.isnull(content).sum() > 0:
                    self.training_medians_[label] = content.median()

        # Learn categorical mappings
        for label, content in X.items():
            if not pd.api.types.is_numeric_dtype(content):
                content_cat = content.astype("category")
                self.category_mappings_[label] = content_cat.cat.categories
        return self

    def transform(self, X):
        X = X.copy()
        # Use a dictionary to collect new columns before joining to avoid fragmentation
        new_columns = {}

        # Process Numeric Columns
        for label, content in X.items():
            if pd.api.types.is_numeric_dtype(content):
                is_missing = pd.isnull(content)
                new_columns[label + "_is_missing"] = is_missing

                if label in self.training_medians_:
                    X[label] = content.fillna(self.training_medians_[label])

        # Process Categorical Columns
        for label, content in X.items():
            if not pd.api.types.is_numeric_dtype(content):
                new_columns[label + "_is_missing"] = pd.isnull(content)
                content_cat = content.astype("category")

                if label in self.category_mappings_:
                    encoded = pd.Categorical(content_cat, categories=self.category_mappings_[label]).codes + 1
                    X[label] = encoded
                else:
                    # Fallback for entirely new categories unseen in training
                    X[label] = pd.Categorical(content_cat).codes + 1

        # Concat all the missing indicator columns at once
        if new_columns:
            new_df = pd.DataFrame(new_columns, index=X.index)
            X = pd.concat([X, new_df], axis=1)

        return X

In [25]:
# Create the pipeline
bulldozer_pipeline = Pipeline([
    ("date_extractor", DateFeatureExtractor()),
    ("imputer_and_encoder", TabularImputerAndEncoder()),
    ("model", RandomForestRegressor(n_estimators=100,
                                    min_samples_leaf=3,
                                    min_samples_split=14,
                                    max_features=0.5,
                                    n_jobs=-1,
                                    max_samples=None,
                                    random_state=42))
])

In [11]:
# Load Raw Training data
df_train = pd.read_csv("data/bluebook-for-bulldozers/Train.csv", low_memory=False)
df_train

,SalesID,SalePrice,MachineID,ModelID,datasource,auctioneerID,YearMade,MachineHoursCurrentMeter,UsageBand,saledate,...,Undercarriage_Pad_Width,Stick_Length,Thumb,Pattern_Changer,Grouser_Type,Backhoe_Mounting,Blade_Type,Travel_Controls,Differential_Type,Steering_Controls
0,1139246,66000,999089,3157,121,3.0,2004,68.0,Low,11/16/2006 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139248,57000,117657,77,121,3.0,1996,4640.0,Low,3/26/2004 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
2,1139249,10000,434808,7009,121,3.0,2001,2838.0,High,2/26/2004 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1139251,38500,1026470,332,121,3.0,2001,3486.0,High,5/19/2011 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139253,11000,1057373,17311,121,3.0,2007,722.0,Medium,7/23/2009 0:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401120,6333336,10500,1840702,21439,149,1.0,2005,NaN,NaN,11/2/2011 0:00,...,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN,NaN
401121,6333337,11000,1830472,21439,149,1.0,2005,NaN,NaN,11/2/2011 0:00,...,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN,NaN
401122,6333338,11500,1887659,21439,149,1.0,2005,NaN,NaN,11/2/2011 0:00,...,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN,NaN
401123,6333341,9000,1903570,21435,149,2.0,2005,NaN,NaN,10/25/2011 0:00,...,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN,NaN


In [13]:
# Separate features and labels
X_train = df_train.drop("SalePrice", axis = 1)
y_train = df_train["SalePrice"]

In [14]:
X_train.shape, y_train.shape

((401125, 52), (401125,))

In [26]:
# FIT THE ENTIRE PIPELINE (Learns dates, medians, categories and trains the model)
print("Training pipeline...")
bulldozer_pipeline.fit(X_train, y_train)

Training pipeline...


,steps,"[('date_extractor', ...), ('imputer_and_encoder', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,14
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,0.5


In [27]:
bulldozer_pipeline.score(X_train, y_train)

0.9512741256122559

In [30]:
# Load Raw Validation Data
X_valid = pd.read_csv("data/bluebook-for-bulldozers/Valid.csv", low_memory=False)

# Predict directly on raw validation data
valid_preds = bulldozer_pipeline.predict(X_valid)
valid_preds

array([39039.75101221, 69523.78482925, 32409.75785392, ...,
       10040.04841496, 12502.56188436, 17500.66563563], shape=(11573,))

In [34]:
# Create an evaluation function
def evaluate_and_predict_valid(X_valid, y_valid, pipeline, output_csv_path="valid_predictions.csv"):
    """
    Evaluates the validation set against the solution set, calculates R2, MAE, and RMSLE, and generates the formatted prediction dataset.
    """

    # Keep a copy of SalesID for the final output
    sales_ids = df_valid["SalesID"].copy()

    # Generate predictions
    print("Generating predictions on validation set...")
    preds = pipeline.predict(X_valid)

    # Build a temporary dataframe to align predictions with true values by SalesID
    df_preds = pd.DataFrame({
        "SalesID": sales_ids,
        "SalePrice_pred": preds
    })

    # Merge with the solution dataframe on SalesID to ensure strict alignment
    merged_eval = df_preds.merge(y_valid, on="SalesID")
    y_true = merged_eval["SalePrice"]
    y_pred = merged_eval["SalePrice_pred"]

    # Calculate metrics (ensure no negative values exist for MSLE calculations)
    y_pred_clipped = np.clip(y_pred, 0, None)

    r2 = r2_score(y_true, y_pred_clipped)
    mae = mean_absolute_error(y_true, y_pred_clipped)
    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred_clipped))

    # print metrics
    print("\n--- Validation Metrics ---")
    print(f"R2 Score: {r2:.4f}")
    print(f"MAE     : {mae:.4f}")
    print(f"RMSLE   : {rmsle:.4f}")

    # Generate and save the final predictions dataset format requested
    final_submission = pd.DataFrame({
        "SalesID": sales_ids,
        "SalePrice": preds
    })

    final_submission.to_csv(output_csv_path, index=False)
    print(f"\nPredictions successfully saved to '{output_csv_path}'")

    return final_submission, {"r2": r2, "mae": mae, "rmsle": rmsle}

In [37]:
submission_df, metrics = evaluate_and_predict_valid(
    X_valid = pd.read_csv("data/bluebook-for-bulldozers/Valid.csv", low_memory=False),
    y_valid = pd.read_csv("data/bluebook-for-bulldozers/ValidSolution.csv", low_memory=False),
    pipeline = bulldozer_pipeline,
    output_csv_path = "valid_predictions.csv"
)

# Preview the first few rows of the generated prediction dataset
print("\nPreview of final prediction dataset:")
print(submission_df.head())

Generating predictions on validation set...

--- Validation Metrics ---
R2 Score: 0.8827
MAE     : 5895.7777
RMSLE   : 0.2431

Predictions successfully saved to 'valid_predictions.csv'

Preview of final prediction dataset:
   SalesID     SalePrice
0  1222837  39039.751012
1  1222839  69523.784829
2  1222841  32409.757854
3  1222843  16223.310272
4  1222845  41386.787766
